# Referee robustness checks
-------

Two referee points are addressed here, each self-contained below.

1. **Fold heterogeneity** - leave-one-fold-out re-pooling of the out-of-fold predictions
   (Tables 1-4) and a seed sweep quantifying retraining variability (Tables 5-6).
2. **The excluded middle and wall-to-wall extrapolation** - where the mapped OGF area sits
   relative to the reference labels (Table 7) and agreement with Schickhofer and Schwarz
   (2019, PRIMOFARO) among unlabelled parcels binned by recorded stand age (Table 8 and
   the figure).


## Fold heterogeneity: leave-one-fold-out and seed sweep

1. Leave-one-fold-out (LOFO) re-pooling.

2. Seed sweep (retraining variability).

Sources: `results/main_nested_cv/` (0 km), `data/cache/performance_matrix_buffered/`
(10 km fixed-HP refits, the arm reported in the performance matrix),
`results/spatial_baseline/` and `results/spatial_buffer/` (coordinate-only control),
`results/seed_sweep/` (seed sweep). Everything downstream of the first cell reads from the
parcel-level score table persisted at `results/fold_sensitivity/oof_parcel_scores.parquet`.


In [ ]:
NOTEBOOK = "014_robustness_checks"

import json

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

from utils.io import ColumnSpec, write_parquet
from utils.paths import get_project_paths
from utils.terminology import FEATURE_SETS, FOLD_IDS, SEED

paths = get_project_paths()

XY = "xy_coords"
CONFIG_LABEL = {**FEATURE_SETS, XY: "Coordinate-only control (x, y)"}
BUFFERS = (0, 10)
CONFIGS = [(fs, km) for km in BUFFERS for fs in [*FEATURE_SETS, XY]]


def _headline_run(fs):
    """Newest complete main nested-CV run for one XGBoost feature set (as in 013)."""
    for run in sorted((paths.results / "main_nested_cv").glob(f"*__xgboost__{fs}"), reverse=True):
        if all((run / f"parcel_predictions_fold{f}.parquet").is_file() for f in FOLD_IDS):
            return run
    raise FileNotFoundError(f"no complete main nested-CV run for {fs!r}")


def _config_frames(fs, km):
    """{fold: out-of-fold parcel predictions} for one (feature set, buffer) configuration."""
    if fs == XY:
        if km == 0:  # the coordinate-only spatial-baseline run
            run = sorted((paths.results / "spatial_baseline").glob(f"*__xgboost__{XY}"))[-1]
            return {
                f: pd.read_parquet(run / f"parcel_predictions_fold{f}.parquet") for f in FOLD_IDS
            }
        # 10 km: the buffered arm of the newest buffered coordinate gap-analysis run.
        runs = [
            r
            for r in sorted((paths.results / "spatial_buffer").glob(f"*__xgboost__{XY}"))
            if (r / "run_arms.json").is_file()
            and json.loads((r / "run_arms.json").read_text(encoding="utf-8"))["arms"]
            == ["buffered"]
        ]
        frames = {}
        for f in FOLD_IDS:
            ck = pd.read_parquet(runs[-1] / "fold_checkpoints" / f"fold{f}_parcel.parquet")
            sub = ck[(ck["arm"] == "buffered") & (ck["gap_km"] == km)]
            frames[f] = sub[
                ["parcel_id", "outer_fold", "y_true", "p_mean", "n_pixels"]
            ].reset_index(drop=True)
        return frames
    run = _headline_run(fs)
    if km == 0:  # the main nested-CV out-of-fold predictions
        return {f: pd.read_parquet(run / f"parcel_predictions_fold{f}.parquet") for f in FOLD_IDS}
    # 10 km: the fixed-HP buffered refits cached by run_performance_matrix (the reported arm).
    cdir = paths.cache / "performance_matrix_buffered" / f"xgboost__{fs}__{run.name}__{km}km"
    return {f: pd.read_parquet(cdir / f"buffered_fold{f}.parquet") for f in FOLD_IDS}


def score_col(fs, km):
    return f"score__{fs}__{km}km"


# One parcel-level table: parcel_id, fold, label, and each configuration's OOF score.
oof = None
for fs, km in CONFIGS:
    frames = _config_frames(fs, km)
    pooled = pd.concat(
        [fr.assign(fold=np.int8(f)) for f, fr in sorted(frames.items())], ignore_index=True
    )
    part = pooled[["parcel_id", "fold", "y_true", "p_mean"]].rename(
        columns={"y_true": "label", "p_mean": score_col(fs, km)}
    )
    if oof is None:
        oof = part
        continue
    merged = oof.merge(part, on="parcel_id", how="inner", suffixes=("", "__check"))
    # Data-integrity checks must survive python -O, so raise rather than assert.
    if not (len(merged) == len(oof) == len(part)):
        raise ValueError(f"{fs} {km} km: parcel set differs from the other configurations")
    if not (merged["fold"] == merged["fold__check"]).all():
        raise ValueError(f"{fs} {km} km: fold assignment differs")
    if not (merged["label"] == merged["label__check"]).all():
        raise ValueError(f"{fs} {km} km: labels differ")
    oof = merged.drop(columns=["fold__check", "label__check"])

oof = oof.sort_values(["fold", "parcel_id"]).reset_index(drop=True)
oof["label"] = oof["label"].astype(np.int8)

OOF_SCHEMA = {
    "parcel_id": ColumnSpec("int64"),
    "fold": ColumnSpec("int8"),
    "label": ColumnSpec("int8"),
    **{score_col(fs, km): ColumnSpec("float64") for fs, km in CONFIGS},
}
oof_path = write_parquet(
    oof, paths.results / "fold_sensitivity" / "oof_parcel_scores.parquet", OOF_SCHEMA
)
print(f"[oof] {len(oof)} parcels x {len(CONFIGS)} configurations -> {oof_path}")

fold_prevalence = oof.groupby("fold")["label"].agg(n_parcels="size", n_ogf="sum", prevalence="mean")
print("\n[folds] parcel counts and OGF prevalence per fold")
print(fold_prevalence.to_string(float_format=lambda v: f"{v:.3f}"))

## Leave-one-fold-out pooled metrics

For each fold *k* the pooled metric is recomputed on the concatenated out-of-fold scores of
the other five folds.

In [ ]:
SCENARIOS = {0: oof, **{k: oof[oof["fold"] != k] for k in FOLD_IDS}}
METRIC_FUNCS = {"pr_auc": average_precision_score, "roc_auc": roc_auc_score}

rows = []
for left_out, sub in SCENARIOS.items():
    y = sub["label"].to_numpy()
    for fs, km in CONFIGS:
        p = sub[score_col(fs, km)].to_numpy()
        for metric, func in METRIC_FUNCS.items():
            rows.append(
                {
                    "left_out_fold": left_out,  # 0 = all folds
                    "feature_set": fs,
                    "buffer_km": km,
                    "metric": metric,
                    "value": float(func(y, p)),
                    "subset_prevalence": float(sub["label"].mean()),
                    "n_parcels": int(len(sub)),
                }
            )
lofo_tidy = pd.DataFrame(rows)


def jackknife_se(values):
    """sqrt((n-1)/n x sum((theta_i - theta_bar)^2)) over the leave-one-out estimates."""
    v = np.asarray(values, dtype=float)
    return float(np.sqrt((len(v) - 1) / len(v) * ((v - v.mean()) ** 2).sum()))


LOO_COLS = [f"excl. fold {k}" for k in FOLD_IDS]


def sensitivity_table(tidy, metric, index_cols, label_of):
    """Rows = index tuples; columns = all folds, the six LOFO estimates, range, jackknife SE."""
    sub = tidy[tidy["metric"] == metric]
    wide = sub.pivot_table(index=index_cols, columns="left_out_fold", values="value", sort=False)
    out = pd.DataFrame(index=wide.index)
    out["all folds"] = wide[0]
    for k in FOLD_IDS:
        out[f"excl. fold {k}"] = wide[k]
    loo = out[LOO_COLS]
    out["range"] = loo.max(axis=1) - loo.min(axis=1)
    out["jackknife SE"] = [jackknife_se(r) for r in loo.to_numpy()]
    out.index = [label_of(i) for i in out.index]
    return out


def prevalence_row(tidy):
    prev = tidy.drop_duplicates("left_out_fold").set_index("left_out_fold")["subset_prevalence"]
    row = {"all folds": prev[0], **{f"excl. fold {k}": prev[k] for k in FOLD_IDS}}
    row["range"] = max(prev[k] for k in FOLD_IDS) - min(prev[k] for k in FOLD_IDS)
    row["jackknife SE"] = np.nan
    return pd.DataFrame([row], index=["OGF prevalence of subset"])


def fmt3(v):
    return "" if pd.isna(v) else f"{v:.3f}"


def show(table, title):
    print(title)
    print(table.map(fmt3).to_string())
    print()


def config_label(key):
    fs, km = key
    return f"{CONFIG_LABEL[fs]} ({km} km)"


lofo_config_pr = pd.concat(
    [
        prevalence_row(lofo_tidy),
        sensitivity_table(lofo_tidy, "pr_auc", ["feature_set", "buffer_km"], config_label),
    ]
)
lofo_config_roc = pd.concat(
    [
        prevalence_row(lofo_tidy),
        sensitivity_table(lofo_tidy, "roc_auc", ["feature_set", "buffer_km"], config_label),
    ]
)
show(lofo_config_pr, "[Table 1] Leave-one-fold-out pooled PR-AUC per configuration")
show(lofo_config_roc, "[Table 2] Leave-one-fold-out pooled ROC-AUC per configuration")

## Paired contrasts on the identical subset


In [ ]:
CONTRAST_PAIRS = [
    ("baseline_conventional_eo", "baseline"),
    ("baseline_alphaearth", "baseline"),
    ("baseline_tessera", "baseline"),
    ("baseline_alphaearth", "baseline_conventional_eo"),
    ("baseline_tessera", "baseline_conventional_eo"),
    ("baseline_tessera", "baseline_alphaearth"),
]
FS_VS_BASELINE = [f"{a} - {b}" for a, b in CONTRAST_PAIRS if b == "baseline"]

vals = lofo_tidy.set_index(["feature_set", "buffer_km", "metric", "left_out_fold"])["value"]
prev = lofo_tidy.drop_duplicates("left_out_fold").set_index("left_out_fold")["subset_prevalence"]
rows = []
for set_a, set_b in CONTRAST_PAIRS:
    for km in BUFFERS:
        for metric in METRIC_FUNCS:
            for left_out in [0, *FOLD_IDS]:
                rows.append(
                    {
                        "contrast": f"{set_a} - {set_b}",
                        "set_a": set_a,
                        "set_b": set_b,
                        "buffer_km": km,
                        "metric": metric,
                        "left_out_fold": left_out,
                        "value": vals[(set_a, km, metric, left_out)]
                        - vals[(set_b, km, metric, left_out)],
                        "subset_prevalence": prev[left_out],
                    }
                )
contrast_tidy = pd.DataFrame(rows)


def contrast_label(key):
    name, km = key
    set_a, set_b = name.split(" - ")
    return f"{CONFIG_LABEL[set_a]} - {CONFIG_LABEL[set_b]} ({km} km)"


contrast_pr = pd.concat(
    [
        prevalence_row(lofo_tidy),
        sensitivity_table(contrast_tidy, "pr_auc", ["contrast", "buffer_km"], contrast_label),
    ]
)
contrast_roc = pd.concat(
    [
        prevalence_row(lofo_tidy),
        sensitivity_table(contrast_tidy, "roc_auc", ["contrast", "buffer_km"], contrast_label),
    ]
)
show(contrast_pr, "[Table 3] Leave-one-fold-out paired contrasts - pooled PR-AUC difference")
show(contrast_roc, "[Table 4] Leave-one-fold-out paired contrasts - pooled ROC-AUC difference")

# Response-letter check 1: pooled over concatenated OOF scores is not the fold mean.
t10 = ("baseline_tessera", 10)
per_fold = np.array(
    [
        average_precision_score(
            oof.loc[oof["fold"] == k, "label"], oof.loc[oof["fold"] == k, score_col(*t10)]
        )
        for k in FOLD_IDS
    ]
)
print(
    f"[note] Baseline + TESSERA at 10 km: pooled PR-AUC = {vals[(*t10, 'pr_auc', 0)]:.3f}, "
    f"mean of per-fold PR-AUCs = {per_fold.mean():.3f} (folds {per_fold.round(3).tolist()}); "
    "concatenation is sensitive to cross-fold calibration and prevalence, so the two differ."
)

# Response-letter check 2: does dropping fold 3 materially move the 10 km TESSERA contrast?
key = contrast_tidy[
    (contrast_tidy["contrast"] == "baseline_tessera - baseline")
    & (contrast_tidy["buffer_km"] == 10)
    & (contrast_tidy["metric"] == "pr_auc")
].set_index("left_out_fold")["value"]
shifts = (key.drop(0) - key[0]).sort_values()
print(
    f"[note] TESSERA - baseline PR-AUC at 10 km: all folds {key[0]:.3f}; "
    f"excl. fold 3 -> {key[3]:.3f} (shift {key[3] - key[0]:+.3f}); "
    f"largest shift from excluding fold {shifts.abs().idxmax()} "
    f"({shifts[shifts.abs().idxmax()]:+.3f})."
)
sign_stable = (
    contrast_tidy[
        contrast_tidy["contrast"].isin(FS_VS_BASELINE) & (contrast_tidy["left_out_fold"] > 0)
    ]["value"]
    > 0
).all()
print(f"[note] all EO-vs-baseline contrasts keep a positive sign in every subset: {sign_stable}")
eo_pairs = contrast_tidy[
    ~contrast_tidy["contrast"].isin(FS_VS_BASELINE) & (contrast_tidy["metric"] == "pr_auc")
]
piv = eo_pairs.pivot_table(index=["contrast", "buffer_km"], columns="left_out_fold", values="value")
straddle = (piv[list(FOLD_IDS)].min(axis=1) < 0) & (piv[list(FOLD_IDS)].max(axis=1) > 0)
print(
    f"[note] pairwise EO-representation PR-AUC contrasts straddling zero across the six "
    f"subsets: {int(straddle.sum())}/{len(straddle)} - near-zero, subset-dependent "
    "differences, consistent with Point 2 (the representations are indistinguishable)."
)

## Seed sweep - retraining variability


In [ ]:
def _is_complete_sweep(run):
    """A sweep is reportable once its metrics table carries at least two seeds."""
    metrics_path = run / "seed_sweep_metrics.csv"
    return metrics_path.is_file() and pd.read_csv(metrics_path)["seed"].nunique() >= 2


sweep_root = paths.results / "seed_sweep"
sweep_runs = (
    sorted(p for p in sweep_root.iterdir() if _is_complete_sweep(p)) if sweep_root.is_dir() else []
)
seed_config_table = seed_contrast_table = None
if not sweep_runs:
    print(
        "[pending] no completed multi-seed run under results/seed_sweep/; "
        "run scripts/run_seed_sweep.py (GPU) and re-execute this cell."
    )
else:
    sweep_run = sweep_runs[-1]
    meta = json.loads((sweep_run / "run_metadata.json").read_text(encoding="utf-8"))
    sw = pd.read_csv(sweep_run / "seed_sweep_metrics.csv")
    swc = pd.read_csv(sweep_run / "seed_sweep_contrasts.csv")
    pm_run = sorted(p for p in (paths.results / "performance_matrix").iterdir() if p.is_dir())[-1]
    matrix = pd.read_csv(pm_run / "performance_matrix.csv")
    fs_ct = pd.concat(
        [
            pd.read_csv(pm_run / "feature_set_contrasts.csv"),
            pd.read_csv(pm_run / "embedding_contrasts.csv"),
        ],
        ignore_index=True,
    )
    print(f"[seed sweep] {sweep_run.name}: seeds {meta['seeds']}, buffers {meta['buffer_kms']} km")
    for check in meta["integrity_check_paper_seed_0km"]:
        print(
            f"[integrity] {check['feature_set']}: paper-seed 0 km refit vs saved predictions, "
            f"max |dp| = {check['max_abs_p_mean_gap']:.3g}"
        )

    def seed_summary(frame, value_col, keys, ref, ref_keys, label_of):
        """Between-seed spread of a pooled metric next to the evaluation bootstrap CI."""
        grouped = frame.groupby(keys)[value_col]
        out = grouped.agg(**{"seed mean": "mean", "between-seed SD": "std"}).join(
            grouped.agg(lambda v: f"{v.min():.3f}-{v.max():.3f}").rename("seed min-max")
        )
        paper = frame[frame["seed"] == SEED].set_index(keys)[value_col]
        out.insert(0, "paper seed", paper)
        ref_indexed = ref.set_index(ref_keys)
        out["bootstrap 95% CI"] = [
            "[{:.3f}, {:.3f}]".format(
                ref_indexed.loc[i, "pr_auc__ci_lo"], ref_indexed.loc[i, "pr_auc__ci_hi"]
            )
            for i in out.index
        ]
        out["n_seeds"] = grouped.size()
        out.index = [label_of(i) for i in out.index]
        for col in ("paper seed", "seed mean", "between-seed SD"):
            out[col] = out[col].map(fmt3)
        return out

    seed_config_table = seed_summary(
        sw,
        "pr_auc__pooled",
        ["feature_set", "buffer_km"],
        matrix[matrix["architecture"] == "xgboost"],
        ["feature_set", "buffer_km"],
        config_label,
    )
    print("\n[Table 5] Pooled PR-AUC per configuration: seed spread vs evaluation bootstrap CI")
    print(seed_config_table.to_string())

    seed_contrast_table = seed_summary(
        swc,
        "pr_auc__diff",
        ["contrast", "buffer_km"],
        fs_ct,
        ["contrast", "buffer_km"],
        contrast_label,
    )
    print("\n[Table 6] Contrast (PR-AUC difference): seed spread vs paired bootstrap CI")
    print(seed_contrast_table.to_string())

In [ ]:
# Save every table as CSV under figures/014_robustness_checks/tables/, then compose the
# one-sentence summary for the response letter.
table_dir = paths.figures / NOTEBOOK / "tables"
table_dir.mkdir(parents=True, exist_ok=True)
_TABLES = {
    "table1_lofo_config_pr_auc": lofo_config_pr,
    "table2_lofo_config_roc_auc": lofo_config_roc,
    "table3_lofo_contrast_pr_auc": contrast_pr,
    "table4_lofo_contrast_roc_auc": contrast_roc,
    "lofo_tidy": lofo_tidy,
    "contrast_tidy": contrast_tidy,
    "table5_seed_sweep_configs": seed_config_table,
    "table6_seed_sweep_contrasts": seed_contrast_table,
}
saved = []
for name, frame in _TABLES.items():
    if isinstance(frame, pd.DataFrame):
        frame.to_csv(table_dir / f"{name}.csv", index=not name.endswith("_tidy"))
        saved.append(name)
print(f"[tables] saved {len(saved)} CSVs to {table_dir}: {', '.join(saved)}")

loo_key = key.drop(0)
sentence = (
    f"Re-pooling the out-of-fold predictions after leaving out each fold in turn moves the "
    f"10 km TESSERA-baseline PR-AUC contrast from {key[0]:.3f} (all folds) to between "
    f"{loo_key.min():.3f} and {loo_key.max():.3f} (the largest shift, "
    f"{shifts[shifts.abs().idxmax()]:+.3f}, from excluding fold {shifts.abs().idxmax()})"
)
sentence += (
    ", and every EO-vs-baseline contrast keeps its sign in all six leave-one-fold subsets "
    "at both buffers"
    if sign_stable
    else "; note that at least one EO-vs-baseline contrast changes sign in some subset"
)
sentence += (
    f", while the subset OGF prevalence spans " f"{prev.drop(0).min():.0%}-{prev.drop(0).max():.0%}"
)
if seed_config_table is not None:
    max_sd = sw.groupby(["feature_set", "buffer_km"])["pr_auc__pooled"].std().max()
    max_csd = swc.groupby(["contrast", "buffer_km"])["pr_auc__diff"].std().max()
    sentence += (
        f"; refitting with {sw['seed'].nunique()} seeds shifts the pooled PR-AUC by at most "
        f"{max_sd:.3f} SD per configuration ({max_csd:.3f} per contrast), well inside the "
        f"block-bootstrap intervals"
    )
sentence += "."
print("\n[summary] " + sentence)

## Model behaviour on unlabelled parcels

In [ ]:
import json

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize
from sklearn.metrics import cohen_kappa_score

from utils.inference import PIXEL_AREA_HA
from utils.paths import get_project_paths
from utils.style import get_figure_size, save_figure, use_publication_style
from utils.terminology import PALETTE_CATEGORICAL, SEMANTIC_COLOURS

use_publication_style()
paths = get_project_paths()

final = gpd.read_file(
    paths.results / "final" / "ogf_labels_predictions.gpkg", layer="ogf_labels_predictions"
)
# The published GeoPackage carries only this study's outputs; the PRIMOFARO (Schickhofer)
# verdict comes from the comparison run it was packaged from.
final_meta = json.loads((paths.results / "final" / "final_outputs_metadata.json").read_text())
comparison = gpd.read_file(
    paths.results / "comparison" / final_meta["comparison_run"] / "product_parcel_comparison.gpkg",
    columns=["parcel_id", "ogf_schickhofer"],
)
final = final.merge(comparison[["parcel_id", "ogf_schickhofer"]], on="parcel_id", how="left")
if final["ogf_schickhofer"].isna().any():
    raise ValueError("comparison run does not cover every published parcel")
labels = gpd.read_file(paths.processed / "vectors" / "labels" / "ogf_reference_labels.gpkg")
labels["parcel_id"] = labels["parcel_id"].astype(np.int64)

parcels = final.merge(labels[["parcel_id", "final_age"]], on="parcel_id", how="left")
if len(parcels) != len(final):
    raise ValueError("stand-age join changed the parcel count")


def _mapped_pixel_area_ha(frame):
    """Area of the mapped 10 m pixels inside each parcel, in hectares.

    The same convention as the headline mapped area (run_final_inference's
    parcel_consistent_area_ha, which sums the mapped pixels of the parcels it classifies as
    OGF), so the totals here reconcile with main Table 3 rather than sitting ~0.05% above it:
    rasterising a parcel boundary to whole cells recovers slightly less than its polygon area
    (149,698 ha against 149,770 ha over every parcel). Parcel ids are burned onto the published
    binary raster's own grid, and only its valid (non-nodata) cells are counted, so a parcel
    outside the mapped domain correctly gets zero.
    """
    binary_path = paths.results / "final" / "ogf_binary_3035_10m.tif"
    for attempt in (1, 2):  # Dropbox-backed raster reads fail intermittently; retry once
        try:
            with rasterio.open(binary_path) as src:
                mapped = src.read(1) != src.nodata
                grid_crs, transform, shape = src.crs, src.transform, (src.height, src.width)
            break
        except Exception as error:
            if attempt == 2:
                raise
            print(f"[retry] re-reading {binary_path.name} after: {error}")
    ids = rasterize(
        list(
            zip(frame.to_crs(grid_crs).geometry, frame["parcel_id"].astype(np.int64), strict=False)
        ),
        out_shape=shape,
        transform=transform,
        fill=-1,
        dtype="int64",
    )
    inside = mapped & (ids >= 0)
    counts = np.bincount(ids[inside], minlength=int(frame["parcel_id"].max()) + 1)
    return counts[frame["parcel_id"].astype(np.int64).to_numpy()] * PIXEL_AREA_HA


parcels["parcel_id"] = parcels["parcel_id"].astype(np.int64)
parcels["area_ha"] = _mapped_pixel_area_ha(parcels)

# Strata: the two labelled extremes, and unlabelled parcels split by recorded stand age.
AGE_EDGES = [85, 100, 120, 140, 160, np.inf]
AGE_LABELS = ["85-99 yr", "100-119 yr", "120-139 yr", "140-159 yr", "160+ yr"]
unlabelled = parcels["ogf_reference_label"].isna()
age_bin = pd.cut(parcels["final_age"], bins=AGE_EDGES, labels=AGE_LABELS, right=False)
parcels["stratum"] = np.select(
    [
        parcels["ogf_reference_label"] == "ogf",
        parcels["ogf_reference_label"] == "non_ogf",
        unlabelled & parcels["final_age"].notna(),
        unlabelled,
    ],
    [
        "Labelled OGF",
        "Labelled non-OGF",
        "Unlabelled, age " + age_bin.astype(str),
        "Unlabelled, no recorded age",
    ],
)
STRATA = [
    "Labelled OGF",
    "Labelled non-OGF",
    *[f"Unlabelled, age {b}" for b in AGE_LABELS],
    "Unlabelled, no recorded age",
]
if set(parcels["stratum"]) != set(STRATA):
    raise ValueError(f"unexpected strata: {sorted(set(parcels['stratum']))}")

no_prediction = parcels["OGF_probability"].isna()
print(
    f"[data] {len(parcels)} parcels; {int(unlabelled.sum())} unlabelled "
    f"({unlabelled.mean():.0%}), of which {int((unlabelled & parcels['final_age'].notna()).sum())} "
    f"carry a recorded stand age; {int(no_prediction.sum())} parcels have no model prediction "
    "(no valid pixels) and are excluded from the rates below."
)
parcels = parcels[~no_prediction].copy()
parcels["ours"] = parcels["OGF_binary"].astype(bool)
parcels["schickhofer"] = parcels["ogf_schickhofer"].astype(bool)

In [ ]:
def stratum_rows(frame):
    rows = {}
    mapped_total = frame.loc[frame["ours"], "area_ha"].sum()
    for stratum in STRATA:
        sub = frame[frame["stratum"] == stratum]
        ogf_area = sub.loc[sub["ours"], "area_ha"].sum()
        rows[stratum] = {
            "parcels": len(sub),
            "area (ha)": sub["area_ha"].sum(),
            "OGF-classified parcels": int(sub["ours"].sum()),
            "OGF-classified share (%)": 100.0 * sub["ours"].mean(),
            "OGF-classified area (ha)": ogf_area,
            "share of mapped OGF area (%)": 100.0 * ogf_area / mapped_total,
            "mean calibrated probability": sub["OGF_probability"].mean(),
        }
    total = {
        "parcels": len(frame),
        "area (ha)": frame["area_ha"].sum(),
        "OGF-classified parcels": int(frame["ours"].sum()),
        "OGF-classified share (%)": 100.0 * frame["ours"].mean(),
        "OGF-classified area (ha)": mapped_total,
        "share of mapped OGF area (%)": 100.0,
        "mean calibrated probability": frame["OGF_probability"].mean(),
    }
    return pd.DataFrame({**rows, "All parcels": total}).T


area_table = stratum_rows(parcels)
print("[Table 7] Deployed OGF classification by labelling stratum")
with pd.option_context("display.float_format", lambda v: f"{v:,.1f}", "display.width", 200):
    print(area_table.to_string())

unl = parcels[parcels["stratum"].str.startswith("Unlabelled")]
unl_share = area_table.loc[
    [s for s in STRATA if s.startswith("Unlabelled")], "share of mapped OGF area (%)"
].sum()
print(
    f"\n[note] {unl_share:.1f}% of the mapped OGF area lies in unlabelled parcels; "
    f"the labelled-OGF stratum contributes "
    f"{area_table.loc['Labelled OGF', 'share of mapped OGF area (%)']:.1f}% and "
    f"the labelled non-OGF stratum "
    f"{area_table.loc['Labelled non-OGF', 'share of mapped OGF area (%)']:.1f}%."
)

## Stratified agreement with Schickhofer and Schwarz (2019) among unlabelled parcels


In [ ]:
# The two labelled strata are carried alongside the unlabelled ones so the excluded
# middle can be read against the extremes the reference labels sample. Rows are built
# from ``parcels`` rather than ``unl``, which for the unlabelled strata is the same subset.
AGREE_STRATA = [
    "Labelled OGF",
    "Labelled non-OGF",
    *[f"Unlabelled, age {b}" for b in AGE_LABELS],
    "Unlabelled, no recorded age",
]


def agreement_row(sub):
    ours, other = sub["ours"], sub["schickhofer"]
    both = (ours & other).sum()
    either = (ours | other).sum()
    # Negative agreement mirrors the positive one onto the non-OGF call: of the parcels
    # called non-OGF by either product (that is, every parcel the two did not both flag),
    # the share called non-OGF by both.
    neither = (~ours & ~other).sum()
    either_non = len(sub) - both
    return {
        "parcels": len(sub),
        "area (ha)": sub["area_ha"].sum(),
        "ours OGF (%)": 100.0 * ours.mean(),
        "PRIMOFARO OGF (%)": 100.0 * other.mean(),
        "both (%)": 100.0 * both / len(sub),
        "overall agreement (%)": 100.0 * (ours == other).mean(),
        "positive agreement (%)": 100.0 * both / either if either else np.nan,
        "negative agreement (%)": 100.0 * neither / either_non if either_non else np.nan,
        "kappa": cohen_kappa_score(ours, other) if ours.nunique() > 1 else np.nan,
        "mean calibrated probability": sub["OGF_probability"].mean(),
    }


agreement_table = pd.DataFrame(
    {
        **{s: agreement_row(parcels[parcels["stratum"] == s]) for s in AGREE_STRATA},
        "All unlabelled": agreement_row(unl),
        "All parcels": agreement_row(parcels),
    }
).T
print("[Table 8] Agreement between the deployed model and PRIMOFARO, by labelling stratum")
with pd.option_context("display.float_format", lambda v: f"{v:,.2f}", "display.width", 200):
    print(agreement_table.to_string())

aged = agreement_table.loc[[f"Unlabelled, age {b}" for b in AGE_LABELS]]
print(
    f"\n[note] across the recorded-age bins the model's OGF rate rises from "
    f"{aged['ours OGF (%)'].iloc[0]:.1f}% (85-99 yr) to {aged['ours OGF (%)'].iloc[-1]:.1f}% "
    f"(>= 160 yr); PRIMOFARO rises from {aged['PRIMOFARO OGF (%)'].iloc[0]:.1f}% to "
    f"{aged['PRIMOFARO OGF (%)'].iloc[-1]:.1f}%; positive agreement moves from "
    f"{aged['positive agreement (%)'].iloc[0]:.1f}% to "
    f"{aged['positive agreement (%)'].iloc[-1]:.1f}%."
)

In [ ]:
# Figure: OGF-classification rates by recorded stand age among unlabelled parcels, the two
# products side by side, with positive agreement overlaid.
OURS_COLOUR = SEMANTIC_COLOURS["ogf"]
PRIMOFARO_COLOUR = PALETTE_CATEGORICAL["blue"]
AGREE_COLOUR = "0.25"

fig_bins = [*[f"Unlabelled, age {b}" for b in AGE_LABELS], "Unlabelled, no recorded age"]
ours_pct = agreement_table.loc[fig_bins, "ours OGF (%)"].to_numpy()
primo_pct = agreement_table.loc[fig_bins, "PRIMOFARO OGF (%)"].to_numpy()
pos_agree = agreement_table.loc[fig_bins, "positive agreement (%)"].to_numpy()
n_parcels = agreement_table.loc[fig_bins, "parcels"].astype(int).to_numpy()
tick_labels = [
    f"{lab}\n(n={n:,})" for lab, n in zip([*AGE_LABELS, "no recorded\nage"], n_parcels, strict=True)
]

x = np.arange(len(fig_bins))
width = 0.38
fig, ax = plt.subplots(figsize=get_figure_size("one_half", aspect=0.55))
ax.bar(x - width / 2, ours_pct, width, color=OURS_COLOUR, label="This study (deployed model)")
ax.bar(
    x + width / 2,
    primo_pct,
    width,
    color=PRIMOFARO_COLOUR,
    label="Schickhofer and Schwarz (2019), PRIMOFARO",
)
ax.plot(
    x,
    pos_agree,
    marker="o",
    markersize=3.5,
    linewidth=1.0,
    color=AGREE_COLOUR,
    label="Positive agreement (both / either)",
)
ax.set_xticks(x, tick_labels)
ax.set_ylabel("Parcels classified as OGF (%)")
ax.set_xlabel("Recorded stand age (unlabelled parcels)")
ax.set_ylim(0, 90)
ax.legend(frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
save_figure(
    fig,
    f"{NOTEBOOK}/unlabelled_age_agreement",
    data=agreement_table.loc[fig_bins].reset_index(names="stratum"),
)
plt.show()

In [ ]:
# Save the tables and draft the response-letter caveat paragraph.
table_dir = paths.figures / NOTEBOOK / "tables"
table_dir.mkdir(parents=True, exist_ok=True)
area_table.to_csv(table_dir / "table7_area_by_stratum.csv", index=True)
agreement_table.to_csv(table_dir / "table8_unlabelled_agreement.csv", index=True)
print(f"[tables] saved 2 CSVs to {table_dir}")

mapped_total = parcels.loc[parcels["ours"], "area_ha"].sum()
aged_rows = [f"Unlabelled, age {b}" for b in AGE_LABELS]
aged_share = area_table.loc[aged_rows, "share of mapped OGF area (%)"].sum()
caveat = (
    f"Of the {mapped_total:,.0f} ha classified as OGF by the deployed model, "
    f"{unl_share:.0f}% lies in parcels that carry no reference label - "
    f"{aged_share:.0f}% "
    f"in unlabelled parcels with a recorded stand age of 85 years or more (the 'excluded "
    f"middle' of older managed stands) and "
    f"{area_table.loc['Unlabelled, no recorded age', 'share of mapped OGF area (%)']:.0f}% in "
    f"unlabelled parcels with no recorded age. Because the reference labels are a "
    f"non-probability sample of the two extremes, the mapped share is a model-based "
    f"prediction under extrapolation to this unvalidated stratum, not a design-based area "
    f"estimate in the sense of Stehman and Foody (2019), and we caveat it accordingly. As "
    f"an independent consistency check, the model's OGF-classification rate among "
    f"unlabelled parcels rises with recorded stand age (from "
    f"{aged['ours OGF (%)'].iloc[0]:.0f}% at 85-99 years to "
    f"{aged['ours OGF (%)'].iloc[-1]:.0f}% at 160 years and above), mirroring the "
    f"independent PRIMOFARO inventory ({aged['PRIMOFARO OGF (%)'].iloc[0]:.0f}% to "
    f"{aged['PRIMOFARO OGF (%)'].iloc[-1]:.0f}% across the same bins), with positive "
    f"agreement between the two products reaching "
    f"{aged['positive agreement (%)'].iloc[-1]:.0f}% in the oldest bin; agreement between "
    f"two independent products is consistency, not validation, and we present it as such."
)
print("\n[caveat draft]\n" + caveat)